# Classification, case study 3 — human-in-the-loop hierarchical classification

Each test image is pushed down the label tree learned by
`python -m hitl_sem.classification.initialize`. An image is classified
automatically only while the patch majority vote is decisive enough
(`conf_thresh`) and few of its patches look anomalous (`anomaly_thresh`);
otherwise the run pauses, shows the anomaly and probability maps, and asks you to
pick the class — or to flag the image as out-of-distribution with **NEW (OOD)**.

Every confirmed decision, automatic or human, updates the classifiers and refits
the anomaly detectors before the next image.

Prerequisites: `classification_models/` populated (shipped, or regenerated by `initialize`) and
`data/classification/embeddings_test/` populated by
`python -m hitl_sem.features --data-folder data/classification/images_test --output-folder data/classification/embeddings_test`.


In [ ]:
%matplotlib inline

from hitl_sem.classification import SequentialActiveLearner

Paths. The session log and the updated models are written under `outputs/` so the published assets in `classification_models/` are left intact.

In [ ]:
TEST_DIR = "./data/classification/images_test"
EMBEDDINGS_DIR = "./data/classification/embeddings_test"
MODELS_DIR = "./classification_models"
DIAGNOSTIC_MAPS_DIR = "./outputs/classification/diagnostics"
SAVE_MODELS_DIR = "./outputs/classification/models_updated"
LOG_PATH = "./outputs/classification/active_learning_logs.csv"

# Ground truth is optional: supplying it turns on test mode, which logs whether
# each decision was right. Leave it as None to run blind.
GROUND_TRUTH_CSV = "./classification_ground_truth_mapping.csv"

In [ ]:
learner = SequentialActiveLearner(
    test_dir=TEST_DIR,
    embedding_dir=EMBEDDINGS_DIR,
    models_dir=MODELS_DIR,
    ground_truth_csv=GROUND_TRUTH_CSV,
    log_path=LOG_PATH,
    save_diagnostics=True,
    diagnostics_dir=DIAGNOSTIC_MAPS_DIR,
    conf_thresh=0.95,     # 95% majority vote required
    anomaly_thresh=0.10,  # at most 10% anomalous patches
    max_buffer=50000      # cap on the Isolation Forest refit buffer
)

learner.run()

Run this once the sequence reports *All images processed*.

In [ ]:
learner.save_models(target_dir=SAVE_MODELS_DIR)

The ablation reported in the paper reads the session log back and compares it
against the image-level and patch-level baselines:

```
python -m hitl_sem.classification.benchmark --al-log outputs/classification/active_learning_logs.csv
```

Omit `--al-log` to reproduce the published numbers from the shipped log.
